# Primorsk — CFAR detector, then promote to MLflow

CFAR is unsupervised, classic way of doing detection on radar systems: a model here is a **parameter set**, not learned weights. We test thresholds on a real chip (with the land/water mask applied, exactly as the pipeline does), then **promote** a chosen config to MLflow as a versioned model. The pipeline runs whatever version you register.

https://en.wikipedia.org/wiki/Constant_false_alarm_rate

Run `setup` first so the water mask exists.

In [ ]:
import sys; sys.path.insert(0, '..')
import os, numpy as np, matplotlib.pyplot as plt
import rasterio, pystac_client, planetary_computer
from rasterio.vrt import WarpedVRT

from pipeline.config import PORT_BBOX
from pipeline.data import list_scenes
from pipeline.detector import detect, to_db
from pipeline.metrics import get_site
from pipeline.models import CFARConfig, config_sha, register, list_versions

STAC = 'https://planetarycomputer.microsoft.com/api/stac/v1'

## 1 · Load one chip and the water mask

In [ ]:
def load_chip(scene_id, bbox):
    cat = pystac_client.Client.open(STAC, modifier=planetary_computer.sign)
    item = list(cat.search(collections=['sentinel-1-grd'], ids=[scene_id]).items())[0]
    href = planetary_computer.sign(item).assets['vv'].href
    with rasterio.open(href) as src:
        with WarpedVRT(src, src_crs=src.gcps[1]) as vrt:
            w = rasterio.windows.from_bounds(*bbox, transform=vrt.transform)
            return vrt.read(1, window=w)

scenes = [s for s in list_scenes(*PORT_BBOX, '2022-05-01', '2022-09-30') if s['coverage'] >= 0.9]
scene = scenes[len(scenes)//2]
print('scene', scene['scene_id'], '| orbit', scene['relative_orbit'])
chip = load_chip(scene['scene_id'], PORT_BBOX)
chip_db = to_db(chip)
print('chip', chip.shape)

In [ ]:
# The land/water mask is a site asset built by setup. detect() drops detections on land.
site = get_site('primorsk')
water = None
if site and site['water_mask_path'] and os.path.exists(site['water_mask_path']):
    with rasterio.open(site['water_mask_path']) as src:
        water = src.read(1).astype(bool)
print('water mask:', water.shape if water is not None else 'NOT FOUND — run setup first')

In [ ]:
vmin, vmax = np.nanpercentile(chip_db, [2, 98])
plt.figure(figsize=(8,5)); plt.imshow(chip_db, cmap='gray', vmin=vmin, vmax=vmax)
plt.title('VV (dB)'); plt.axis('off'); plt.show()

## 2 · CFAR threshold testing

`pfa` (probability of false alarm) is the main knob: lower = stricter, higher = looser. The water mask is passed in, so only detections over water count — same as the pipeline.

In [ ]:
for pfa in [1e-6, 1e-4, 1e-2]:
    dets = detect(chip_db, water_mask=water, pfa=pfa)
    print(f'pfa={pfa:>7}: {len(dets)} detections')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, pfa in zip(axes, [1e-6, 1e-4, 1e-2]):
    dets = detect(chip_db, water_mask=water, pfa=pfa)
    ax.imshow(chip_db, cmap='gray', vmin=vmin, vmax=vmax)
    if dets:
        ax.scatter([c for _,c,_ in dets], [r for r,_,_ in dets], c='red', s=25, marker='x')
    ax.set_title(f'pfa={pfa} — {len(dets)} detections'); ax.axis('off')
plt.tight_layout(); plt.show()

## 3 · Promote a config to MLflow

Pick a config and register it. The version is a content sha of the config, so re-registering the same params is a no-op. This is exactly what `setup` does for the baselines; here you do it by hand.

In [ ]:
cfg = CFARConfig(pfa=0.05)   # change this and re-run to register a different model
print('config sha:', config_sha(cfg))
sha = register(cfg)
print('registered cfar version:', sha)
list_versions()

## 4 · Run the pipeline with it

The `monitoring` deployment takes a `model_version`. Pass the sha above (Prefect UI, or below) and it processes the window with that model, writing results keyed on `(site, model_version, scene)`. Switch the version in Grafana to compare models.

In [ ]:
# from pipeline.flows import process_day
# process_day(model_version=sha, day='2022-05-26')   # one day, locally